# 🦀 Day 6: Command-Line Arguments

Today: building CLI tools!
- `std::env::args()` — basic argument parsing
- `clap` crate — professional CLI parsing
- Building a text processing tool

⚠️ **Note:** CLI args don't work in Jupyter. This notebook teaches concepts —
practice by building a real Cargo project!

---

## 🖥️ Basic Args with `std::env`

```rust
// In main.rs:
use std::env;

fn main() {
    let args: Vec<String> = env::args().collect();
    
    println!("Program: {}", args[0]);    // Always the program name
    
    if args.len() < 2 {
        println!("Usage: {} <name>", args[0]);
        return;
    }
    
    println!("Hello, {}!", args[1]);
}
```

```bash
$ cargo run -- Alice
Hello, Alice!
```

## 🏗️ Simulating CLI Parsing

Let's build arg parsing logic we can test in the notebook:

In [ ]:
#[derive(Debug)]
struct CliArgs {
    command: String,
    filename: Option<String>,
    verbose: bool,
    count: u32,
}

fn parse_args(args: &[&str]) -> Result<CliArgs, String> {
    if args.is_empty() {
        return Err("No command provided. Use: count, reverse, upper".into());
    }

    let command = args[0].to_string();
    let mut filename = None;
    let mut verbose = false;
    let mut count = 1;

    let mut i = 1;
    while i < args.len() {
        match args[i] {
            "-v" | "--verbose" => verbose = true,
            "-n" | "--count" => {
                i += 1;
                if i >= args.len() {
                    return Err("--count requires a value".into());
                }
                count = args[i].parse()
                    .map_err(|_| format!("Invalid count: {}", args[i]))?;
            }
            "-f" | "--file" => {
                i += 1;
                if i >= args.len() {
                    return Err("--file requires a path".into());
                }
                filename = Some(args[i].to_string());
            }
            other => {
                if other.starts_with('-') {
                    return Err(format!("Unknown flag: {}", other));
                }
                filename = Some(other.to_string());
            }
        }
        i += 1;
    }

    Ok(CliArgs { command, filename, verbose, count })
}

// Test parsing
println!("{:?}\n", parse_args(&["count", "-v", "-n", "5", "file.txt"]));
println!("{:?}\n", parse_args(&["reverse", "--file", "input.txt"]));
println!("{:?}\n", parse_args(&["upper"]));
println!("{:?}\n", parse_args(&[]));  // Error
println!("{:?}", parse_args(&["count", "--unknown"]));  // Error

## 📦 The `clap` Crate

`clap` is the standard CLI parsing crate. It handles:
- Arguments, flags, subcommands
- Help text generation
- Validation and error messages

### Using `clap` with Derive (Recommended)

```rust
// In Cargo.toml:
// [dependencies]
// clap = { version = "4", features = ["derive"] }

use clap::Parser;

/// A simple text processing tool
#[derive(Parser, Debug)]
#[command(name = "textool", version = "1.0")]
struct Args {
    /// Input file to process
    filename: String,
    
    /// Operation: count, reverse, upper, lower
    #[arg(short, long, default_value = "count")]
    operation: String,
    
    /// Show verbose output
    #[arg(short, long)]
    verbose: bool,
    
    /// Output file (defaults to stdout)
    #[arg(short = 'o', long)]
    output: Option<String>,
}

fn main() {
    let args = Args::parse();
    println!("File: {}", args.filename);
    println!("Op: {}", args.operation);
    println!("Verbose: {}", args.verbose);
}
```

Auto-generated help:
```bash
$ textool --help
A simple text processing tool

Usage: textool [OPTIONS] <FILENAME>

Arguments:
  <FILENAME>  Input file to process

Options:
  -o, --operation <OP>    Operation [default: count]
  -v, --verbose           Show verbose output
  -o, --output <FILE>     Output file
  -h, --help              Print help
  -V, --version           Print version
```

### Subcommands with `clap`

```rust
use clap::{Parser, Subcommand};

#[derive(Parser)]
#[command(name = "notes")]
struct Cli {
    #[command(subcommand)]
    command: Commands,
}

#[derive(Subcommand)]
enum Commands {
    /// Add a new note
    Add {
        /// The note text
        text: String,
    },
    /// List all notes  
    List {
        /// Show only last N notes
        #[arg(short, long)]
        last: Option<usize>,
    },
    /// Delete a note by ID
    Delete {
        /// Note ID to delete
        id: usize,
    },
}

fn main() {
    let cli = Cli::parse();
    match cli.command {
        Commands::Add { text } => println!("Adding: {}", text),
        Commands::List { last } => println!("Listing (last {:?})", last),
        Commands::Delete { id } => println!("Deleting #{}", id),
    }
}
```

```bash
$ notes add "Learn Rust CLI"
$ notes list --last 5
$ notes delete 3
```

## 🔧 Text Processing Functions

In [ ]:
// Functions we'd use in our CLI tool

fn count_stats(text: &str) -> (usize, usize, usize, usize) {
    let lines = text.lines().count();
    let words = text.split_whitespace().count();
    let chars = text.chars().count();
    let bytes = text.len();
    (lines, words, chars, bytes)
}

fn reverse_words(text: &str) -> String {
    text.lines()
        .map(|line| {
            line.split_whitespace()
                .rev()
                .collect::<Vec<&str>>()
                .join(" ")
        })
        .collect::<Vec<String>>()
        .join("\n")
}

fn to_title_case(text: &str) -> String {
    text.split_whitespace()
        .map(|word| {
            let mut chars = word.chars();
            match chars.next() {
                None => String::new(),
                Some(c) => c.to_uppercase().to_string() + &chars.as_str().to_lowercase(),
            }
        })
        .collect::<Vec<String>>()
        .join(" ")
}

let sample = "hello world from rust\nthis is a test\nrust is awesome";

let (lines, words, chars, bytes) = count_stats(sample);
println!("📊 Stats: {} lines, {} words, {} chars, {} bytes\n", lines, words, chars, bytes);

println!("🔄 Reversed words:");
println!("{}\n", reverse_words(sample));

println!("📝 Title Case:");
println!("{}", to_title_case(sample));

## 🏋️ Exercises

In [ ]:
// Exercise 1: Write a function find_and_replace(text, find, replace)
// that works on multi-line text and reports how many replacements it made

// YOUR CODE HERE


In [ ]:
// Exercise 2: Write a function that extracts all unique words from text
// sorted alphabetically (case-insensitive)

// YOUR CODE HERE


## 🎯 Key Takeaways

1. `std::env::args()` for basic CLI arg parsing
2. `clap` crate (with derive) for professional CLIs
3. Subcommands, flags, options, and validation for free
4. Always provide `--help` and meaningful error messages
5. Combine file I/O + CLI args for powerful tools

---
**Tomorrow:** Mini-project — CLI Word Counter (like `wc`)! 🧮